In [1]:
!wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt

--2026-03-05 11:46:43--  https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.02s   

2026-03-05 11:46:44 (9.31 MB/s) - ‘names.txt’ saved [228145/228145]



In [2]:
with open("names.txt", "r") as f:
  words = [line.strip() for line in f]

words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

##### Create a 27*27 matrix for ([.a-z]) where each cell entry is freq count of bigrams
##### Once done, sample a cell from first row (i.e. '.' as start) and continue sampling from different rows until '.' is encountered again

In [4]:
import torch

In [8]:
P = torch.ones(27,27, dtype=torch.float64)
P.shape

torch.Size([27, 27])

In [6]:
P.dtype

torch.float64

In [9]:
for word in words:
  word = f".{word}."
  for char1, char2 in zip(word, word[1:]):
    char1_idx = 0 if char1 == "." else ord(char1)-ord('a')+1
    char2_idx = 0 if char2 == "." else ord(char2)-ord('a')+1
    P[char1_idx, char2_idx] += 1

In [10]:
torch.set_printoptions(sci_mode=False)
P
# Original
#        0.0000, 0.1377, 0.0408, 0.0481, 0.0528, 0.0478, 0.0130, 0.0209, 0.0273,
#        0.0184, 0.0756, 0.0925, 0.0491, 0.0792, 0.0358, 0.0123, 0.0161, 0.0029,
#        0.0512, 0.0642, 0.0408, 0.0024, 0.0117, 0.0096, 0.0042, 0.0167, 0.0290

tensor([[   1., 4411., 1307., 1543., 1691., 1532.,  418.,  670.,  875.,  592.,
         2423., 2964., 1573., 2539., 1147.,  395.,  516.,   93., 1640., 2056.,
         1309.,   79.,  377.,  308.,  135.,  536.,  930.],
        [6641.,  557.,  542.,  471., 1043.,  693.,  135.,  169., 2333., 1651.,
          176.,  569., 2529., 1635., 5439.,   64.,   83.,   61., 3265., 1119.,
          688.,  382.,  835.,  162.,  183., 2051.,  436.],
        [ 115.,  322.,   39.,    2.,   66.,  656.,    1.,    1.,   42.,  218.,
            2.,    1.,  104.,    1.,    5.,  106.,    1.,    1.,  843.,    9.,
            3.,   46.,    1.,    1.,    1.,   84.,    1.],
        [  98.,  816.,    1.,   43.,    2.,  552.,    1.,    3.,  665.,  272.,
            4.,  317.,  117.,    1.,    1.,  381.,    2.,   12.,   77.,    6.,
           36.,   36.,    1.,    1.,    4.,  105.,    5.],
        [ 517., 1304.,    2.,    4.,  150., 1284.,    6.,   26.,  119.,  675.,
           10.,    4.,   61.,   31.,   32.,  379.,   

#### Computing probabilities instead

In [11]:
P /= P.sum(dim=1, keepdim=True)

In [27]:
P

tensor([[0.0000, 0.1376, 0.0408, 0.0481, 0.0527, 0.0478, 0.0130, 0.0209, 0.0273,
         0.0185, 0.0756, 0.0925, 0.0491, 0.0792, 0.0358, 0.0123, 0.0161, 0.0029,
         0.0512, 0.0641, 0.0408, 0.0025, 0.0118, 0.0096, 0.0042, 0.0167, 0.0290],
        [0.1958, 0.0164, 0.0160, 0.0139, 0.0308, 0.0204, 0.0040, 0.0050, 0.0688,
         0.0487, 0.0052, 0.0168, 0.0746, 0.0482, 0.1604, 0.0019, 0.0024, 0.0018,
         0.0963, 0.0330, 0.0203, 0.0113, 0.0246, 0.0048, 0.0054, 0.0605, 0.0129],
        [0.0430, 0.1205, 0.0146, 0.0007, 0.0247, 0.2455, 0.0004, 0.0004, 0.0157,
         0.0816, 0.0007, 0.0004, 0.0389, 0.0004, 0.0019, 0.0397, 0.0004, 0.0004,
         0.3155, 0.0034, 0.0011, 0.0172, 0.0004, 0.0004, 0.0004, 0.0314, 0.0004],
        [0.0275, 0.2293, 0.0003, 0.0121, 0.0006, 0.1551, 0.0003, 0.0008, 0.1869,
         0.0764, 0.0011, 0.0891, 0.0329, 0.0003, 0.0003, 0.1071, 0.0006, 0.0034,
         0.0216, 0.0017, 0.0101, 0.0101, 0.0003, 0.0003, 0.0011, 0.0295, 0.0014],
        [0.0936, 0.2361,

In [12]:
P[0].sum()

tensor(1., dtype=torch.float64)

In [13]:
P[0].shape, P[0].dim()

(torch.Size([27]), 1)

##### Sample from a multinomial distribution (?)

In [14]:
g = torch.Generator().manual_seed(2147483647)

gen_num_words = 5

for _ in range(gen_num_words):
  row_idx = 0
  out = []

  while True:
    idx = torch.multinomial(P[row_idx], num_samples=1, replacement=True, generator=g)
    row_idx = idx.item()
    if row_idx == 0:
      break
    out.append(chr(ord('a')+row_idx-1))
  print("".join(out))

cexze
momasurailezitynn
konimittain
llayn
ka


#### How to calculate loss :
##### We can imagine that P is the model weight matrix
So based on that, we can iterate over bigrams and calculate negative log likelihood

In [15]:

n = 0
nll = 0

for word in words:
  word = f".{word}."
  for char1, char2 in zip(word, word[1:]):
    char1_idx = 0 if char1 == "." else ord(char1)-ord('a')+1
    char2_idx = 0 if char2 == "." else ord(char2)-ord('a')+1
    nll += torch.log(P[char1_idx, char2_idx]) # This means that for char1 we are considering char2 as prediction
    n += 1 # Counting each bigram

nll *= -1
print(f"{nll=}")

print(f"Avg nll {nll.item()/n}") # This shows avg NLL considering freq matrix approach as one of the models

nll=tensor(560001.8832, dtype=torch.float64)
Avg nll 2.454576820116076


#### Training a model and doing gradient descent

##### APPROACH
Create xs, ys as char1,char2 - One hot encoded

Choose a W matrix [27,27]
- 27 inputs to do one hot encoding of each char into [.a-z]
- 27 outputs to predict a character

##### LOSS

So, we get logits post forward pass, we do a softmax
we get probabilities y_pred of [1,27]
Calculate loss as -log(ys * y_pred)

Do loss backward
and update W


In [16]:
def char_to_idx(c):
  return 0 if c == "." else (ord(c) - ord('a') + 1)

def idx_to_char(idx):
  return '.' if idx == 0 else chr(ord('a') + idx - 1)

In [17]:
idx_to_char(26)

'z'

In [18]:
# Prepare Dataset
X, Y = [], []
for word in words:
  word = f".{word}."
  for char1, char2 in zip(word, word[1:]):
    X.append(char_to_idx(char1))
    Y.append(char_to_idx(char2))

X = torch.tensor(X)
Y = torch.tensor(Y)
Y, Y.shape

(tensor([ 5, 13, 13,  ..., 26, 24,  0]), torch.Size([228146]))

In [20]:
xenc = F.one_hot(X, num_classes=27)
xenc = xenc.double()
xenc.shape, xenc.dtype

(torch.Size([228146, 27]), torch.float64)

In [21]:
yenc = F.one_hot(Y, num_classes=27).double()
yenc.shape, yenc.dtype

(torch.Size([228146, 27]), torch.float64)

In [23]:
W = torch.randn(27,27, dtype=torch.float64, requires_grad=True)
W

tensor([[-0.5021, -0.4814, -0.2584,  0.2878,  1.2934, -0.7555,  1.7123,  0.6415,
          1.0053, -0.4193,  1.1039,  0.2501,  0.7091, -0.5671, -3.4917, -0.5283,
          0.1744,  0.1799,  0.0023,  0.1795, -1.5032,  0.0286,  1.0257,  0.0484,
         -0.4301, -2.4440, -1.1760],
        [-0.2113, -1.9557, -0.9900,  0.3103,  0.3087, -1.0255, -0.5655, -0.8449,
         -0.6999, -0.4839, -0.5814,  0.1200,  1.7314, -0.4619, -0.4965, -0.1916,
          0.9054,  0.3061, -0.2027, -0.7471, -0.7512, -1.0598,  0.2427,  0.8302,
          0.0481, -0.7505, -0.3611],
        [ 2.0763,  0.1470, -1.0212, -0.5538, -0.2052, -0.9329,  0.0252, -0.8213,
         -1.4948,  1.8247, -1.2194,  0.8864,  1.5295, -1.2353,  0.2164, -0.2116,
         -0.6088, -0.6550, -1.2648, -1.1514, -1.4071, -0.7445,  0.3176,  0.6186,
         -1.4097,  0.3204, -0.3982],
        [-0.1199,  0.9563,  1.3357, -1.2635,  0.5824,  1.3356,  1.7004,  0.7139,
         -1.0196, -0.0585, -0.7708, -0.4872,  0.3714,  2.1396, -1.1813, -0.8699

In [24]:
xenc.dtype, W.dtype

(torch.float64, torch.float64)

In [25]:
## Creating W
step = 100
runs = 100

for run in range(runs):
  # Forward pass
  logits = xenc @ W #[nin, 27]
  #probs = torch.nn.Softmax(logits, dim=1)
  log_probs = F.log_softmax(logits, dim=1)

  loss = -(yenc * log_probs).sum(dim=1).mean()

  W.grad = None
  loss.backward()

  W.data -= step * W.grad

  print(f"Run: {run+1} Loss: {loss}")



Run: 1 Loss: 3.8539211095670773
Run: 2 Loss: 3.20052879218472
Run: 3 Loss: 2.9663837623888396
Run: 4 Loss: 2.837183786397822
Run: 5 Loss: 2.7766379863415716
Run: 6 Loss: 2.713424557423867
Run: 7 Loss: 2.6843412985340516
Run: 8 Loss: 2.6484429698223075
Run: 9 Loss: 2.639095652743976
Run: 10 Loss: 2.6117966613522774
Run: 11 Loss: 2.607140080713033
Run: 12 Loss: 2.5880082462690157
Run: 13 Loss: 2.589663308730286
Run: 14 Loss: 2.5710052399190766
Run: 15 Loss: 2.5725323239110938
Run: 16 Loss: 2.5590244849086865
Run: 17 Loss: 2.5642840389285406
Run: 18 Loss: 2.5487226259798788
Run: 19 Loss: 2.552428339573025
Run: 20 Loss: 2.541577886501981
Run: 21 Loss: 2.548627035676477
Run: 22 Loss: 2.5343292457177076
Run: 23 Loss: 2.5390386923923987
Run: 24 Loss: 2.5299160588820855
Run: 25 Loss: 2.5382324049188885
Run: 26 Loss: 2.5244443102069116
Run: 27 Loss: 2.52963371237645
Run: 28 Loss: 2.5218885210326123
Run: 29 Loss: 2.5312191415180894
Run: 30 Loss: 2.517457116028522
Run: 31 Loss: 2.52274285059413
R

In [ ]:
W

tensor([[-3.1239,  1.7096,  0.4916,  0.6580,  0.7497,  0.6508, -0.6540, -0.1789,
          0.0892, -0.3034,  1.1100,  1.3117,  0.6773,  1.1568,  0.3607, -0.7111,
         -0.4416, -2.1145,  0.7191,  0.9455,  0.4932, -2.2276, -0.7582, -0.9629,
         -1.7813, -0.4034,  0.1503],
        [ 2.5439,  0.0619,  0.0344, -0.1067,  0.6910,  0.2811, -1.3583, -1.1353,
          1.4972,  1.1511, -1.1037,  0.0833,  1.5779,  1.1413,  2.3441, -1.9707,
         -1.7721, -2.0074,  1.8336,  0.7615,  0.2739, -0.3174,  0.4681, -1.1964,
         -1.0558,  1.3683, -0.1843],
        [ 0.8967,  2.1793, -0.4145, -0.3543,  0.3400,  2.9206, -1.2666, -0.4113,
         -0.1289,  1.7486, -0.8117, -0.6430,  0.6847, -0.5139, -0.5101,  0.7742,
         -0.8458, -0.6186,  3.1767, -0.2102, -1.2181, -0.4543, -0.6927, -1.6372,
         -1.1763,  0.6031, -0.6965],
        [-0.0796,  2.5273, -1.2048, -1.0249, -1.1207,  2.1256, -2.3257, -1.2186,
          2.3176,  1.3728, -1.9518,  1.5337,  0.3904, -0.9918, -1.0854,  1.7377

In [26]:
# Do a generation as well
# Starting with .
for _ in range(5):
  idx = 0
  out = []
  while True:
    # Forward pass
    xs_enc = F.one_hot(torch.tensor([idx]), num_classes=27).double() #[1,27]
    logits = xs_enc @ W #[1, 27]
    probs = F.softmax(logits, dim=1)
    idx = torch.multinomial(probs, num_samples=1, replacement=True, generator=g)
    #idx = probs.argmax(dim=1)
    #print(f"idx: {idx}")
    #print(f"probs: {probs}")
    c = idx_to_char(idx)
    if c == ".":
      break
    else:
      out.append(c)
  print("".join(out))
  #print("New word\n")


da
staiyaubrtthrigotai
moliellavo
ke
tedaren


### Questions


*   Why multinomial sampling ? (because num_classes=27)
*   Why argmax just gives 'a' as word output every time ?
    Ans: If i inspect matrix P, the prob for a followed by . is highest in first and second rows. Therefore even trained model comes close to those numbers and argmax would just generate that and that's why we need sampling for new answers
*   Why my numbers don't match ?



In [28]:
## Trying with regularization loss

## Creating W
W = torch.randn(27,27, dtype=torch.float64, requires_grad=True)

step = 100
runs = 100

for run in range(runs):
  # Forward pass
  logits = xenc @ W #[nin, 27]
  #probs = torch.nn.Softmax(logits, dim=1)
  log_probs = F.log_softmax(logits, dim=1)

  loss = -(yenc * log_probs).sum(dim=1).mean() + 0.1 * (W**2).mean()
  # Meaning of adding regularization here:
  # The second term tries to bring W to zero i.e. uniform distribution i.e. equal prob OR (N+10000000 in matrix)
  # But the first term wants to bring the probabilities higher so that log loss is close to zero
  # That means there is a tension between 1st and 2nd term
  # So the distribution doesn't become totally matching training data nor becomes totally uniform

  W.grad = None
  loss.backward()

  W.data -= step * W.grad

  print(f"Run: {run+1} Loss: {loss}")

Run: 1 Loss: 3.8230110897242473
Run: 2 Loss: 3.1715871552801502
Run: 3 Loss: 2.957923268283278
Run: 4 Loss: 2.8425710147213157
Run: 5 Loss: 2.780922568162332
Run: 6 Loss: 2.7398444761485488
Run: 7 Loss: 2.730682656065579
Run: 8 Loss: 2.6992330984112356
Run: 9 Loss: 2.697588548568918
Run: 10 Loss: 2.67672785906241
Run: 11 Loss: 2.676422902772857
Run: 12 Loss: 2.6582190888846
Run: 13 Loss: 2.662375792631057
Run: 14 Loss: 2.6486329065251804
Run: 15 Loss: 2.654033364264614
Run: 16 Loss: 2.6398831221889565
Run: 17 Loss: 2.645846415526324
Run: 18 Loss: 2.634891188265766
Run: 19 Loss: 2.6423935731478405
Run: 20 Loss: 2.6299075611578804
Run: 21 Loss: 2.636705792299554
Run: 22 Loss: 2.627093865710453
Run: 23 Loss: 2.6356329262822165
Run: 24 Loss: 2.6239018307143835
Run: 25 Loss: 2.631178126208944
Run: 26 Loss: 2.6223267579564267
Run: 27 Loss: 2.6314729283216383
Run: 28 Loss: 2.620091725265296
Run: 29 Loss: 2.6276353535254198
Run: 30 Loss: 2.6192807888214866
Run: 31 Loss: 2.628821942902416
Run: 

In [30]:
# Do a generation as well
# Starting with .
for _ in range(5):
  idx = 0
  out = []
  while True:
    # Forward pass
    xs_enc = F.one_hot(torch.tensor([idx]), num_classes=27).double() #[1,27]
    logits = xs_enc @ W #[1, 27]
    probs = F.softmax(logits, dim=1)
    idx = torch.multinomial(probs, num_samples=1, replacement=True, generator=g)
    #idx = probs.argmax(dim=1)
    #print(f"idx: {idx}")
    #print(f"probs: {probs}")
    c = idx_to_char(idx)
    if c == ".":
      break
    else:
      out.append(c)
  print("".join(out))
  #print("New word\n")


wil
pyawaisan
jar
din
kezkm
